# 📊 Lab 4: Close the Round-Trip

## What this lab does:

In **Lab 3**, the app wrote operator activity into the Postgres **operational tables**. Because you enabled **Change Data Feed (CDF)** in **Lab 2**, every one of those writes streams straight back into Unity Catalog as an `lb_<table>_history` Delta table — so operational activity is instantly available for analytics.

## What you'll learn:

* How to read the `lb_*_history` **change-history** tables that CDF produces
* How to join operational writes back to the original alerts to compute a real KPI (**time-to-fix / MTTR**)
* How the full loop closes: **UC → Lakebase → app → CDF → back to UC**, all on one governed platform

## Time to Complete: ~5-10 minutes

---

➡️ **Run the cells in order.** Everything here is read-only analytics (the teardown at the end is turned off by default).

## 🔁 The Round-Trip, End to End

```
Unity Catalog (catalog_workshop.schema_{you})     ← Lab 1 analytical source
        ↓ snapshot sync
Lakebase Postgres (databricks_postgres)
   • lakebase_{you} schema  → read-only reference replicas
   • public schema          → operational tables the app writes  ← Lab 3
        ↓ Change Data Feed (~15-30s)
Unity Catalog (catalog_workshop.lakebase_{you})
   • lb_maintenance_actions_history
   • lb_work_orders_history
   • lb_quality_checks_history
   • lb_operator_notes_history
```

The `lb_*_history` tables land in **`catalog_workshop.lakebase_{your_name}`** — the same schema that holds the synced reference replicas.

## Prerequisites

⚠️ Before running this lab:
- **Lab 3** deployed the app and you **logged at least one maintenance action** in it.
- **CDF is running** (Lab 2, Step 10) — give it ~15-30s after your last write for changes to appear.

## 🔍 See the History Tables CDF Produced

Change Data Feed writes one Delta table per operational table, named `lb_<table>_history`. Each row is a captured change event with a `_change_type` column (`insert`, `update_postimage`, `update_preimage`, `delete`) plus a commit version and timestamp — so you get the full change history, not just the latest state.

### What you'll learn:
* How the synced schema (`lakebase_{your_name}`) holds **both** the read-only reference replicas **and** the CDF history tables

In [0]:
import re

# Derive your schema names the same way Lab 1 & Lab 2 did (email -> slug)
CATALOG = "catalog_workshop"
user = spark.sql("SELECT current_user()").first()[0]
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())
LAKEBASE_SCHEMA = f"lakebase_{slug}"   # synced replicas + CDF history land here

print(f"📂 Reading CDF history from: {CATALOG}.{LAKEBASE_SCHEMA}\n")

# The lb_*_history tables are created by CDF (Lab 2, Step 10)
display(
    spark.sql(f"SHOW TABLES IN {CATALOG}.{LAKEBASE_SCHEMA}")
         .filter("tableName LIKE 'lb_%_history'")
)

## 📖 Read What the Technician Did — from Databricks SQL

Every action logged in the app (Lab 3) flows into `lb_maintenance_actions_history` within ~15-30s. Reading it here proves the **round-trip**: an operational write, captured into the lakehouse by CDF, with no hand-written ETL. 🎉

> **Empty result?** Open the app, log an action, wait ~15s, then re-run — CDF flushes in ~15s batches.
> The `_change_type` column lets you see inserts, updates, and deletes over time.

In [0]:
# Latest change events first. _change_type shows insert / update_postimage / delete.
display(spark.sql(f"""
    SELECT *
    FROM {CATALOG}.{LAKEBASE_SCHEMA}.lb_maintenance_actions_history
    ORDER BY performed_at DESC
"""))

## ⏱️ The Metric the Data Team Gets Back: Time-to-Fix

Join the **completed** actions back to the original alerts (`maintenance_tickets`) to compute how long each fix took. Everything is in one schema — the synced `machines` / `maintenance_tickets` replicas and the CDF history — so it's a plain join.

**💡 What just happened?** The technician's action, captured operationally in the app, is instantly analytics-ready. This is the MTTR / time-to-fix metric that would feed an OEE report or retrain the vibration-based failure model. The loop is closed.

In [0]:
# hours_to_fix = completed_at (operational write) - opened_at (original ticket)
display(spark.sql(f"""
    SELECT h.machine_id, m.model, h.performed_by, h.action_type, h.description,
           round((unix_timestamp(h.completed_at) - unix_timestamp(t.opened_at)) / 3600.0, 1) AS hours_to_fix
    FROM {CATALOG}.{LAKEBASE_SCHEMA}.lb_maintenance_actions_history h
    JOIN {CATALOG}.{LAKEBASE_SCHEMA}.machines m              ON m.machine_id = h.machine_id
    LEFT JOIN {CATALOG}.{LAKEBASE_SCHEMA}.maintenance_tickets t ON t.ticket_id = h.ticket_id
    WHERE h.status = 'completed'
    ORDER BY h.completed_at DESC
"""))

## 📊 Change History Across Every Operational Table

CDF captures `INSERT` / `UPDATE` / `DELETE` for **all four** operational tables, not just maintenance actions. That gives you audit trails, SCD-style analytics, or triggers for downstream Lakeflow jobs — all from Delta, with no bespoke connector to maintain.

### Going further
* Build a one-page AI/BI dashboard over this data — see [`docs/dashboard.md`](../docs/dashboard.md).

In [0]:
# How many change events has CDF captured per operational table?
history_tables = [
    "lb_maintenance_actions_history",
    "lb_work_orders_history",
    "lb_quality_checks_history",
    "lb_operator_notes_history",
]
for t in history_tables:
    try:
        n = spark.table(f"{CATALOG}.{LAKEBASE_SCHEMA}.{t}").count()
        print(f"✅ {t:35s} {n:,} change events")
    except Exception as e:
        print(f"⏳ {t:35s} not found yet ({str(e).splitlines()[0][:50]})")

## 🧹 Teardown (optional)

To avoid lingering cost, you can remove everything you created (Lakebase scales to zero when idle, so it's cheap to leave if you'll come back).

⚠️ **This is destructive.** It's turned **off** by default so *Run all* is safe. To use it:
1. **First stop CDF** in the Lakebase UI: your project ▸ **Change Data Feed** ▸ **Stop**.
2. Set `RUN_TEARDOWN = True` in the next cell and run it.

It deletes: the app, the synced tables, the `lakebase_{your_name}` schema (incl. the `lb_*_history` tables), and the Lakebase project (which drops the Postgres operational tables).

In [0]:
# ⚠️ DESTRUCTIVE. Off by default. Stop CDF in the UI first, then set RUN_TEARDOWN = True.
RUN_TEARDOWN = False

if RUN_TEARDOWN:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    app = ("lb-workshop-" + slug)[:30].rstrip("-")

    # find your Lakebase project
    PROJECT = None
    for i in range(1, 21):
        c = f"lakebase-ws-{slug}-{i}"
        try:
            w.postgres.get_project(name=f"projects/{c}"); PROJECT = c; break
        except Exception:
            continue

    # 1) delete the app
    try: w.apps.delete(name=app)
    except Exception as e: print("app:", e)

    # 2) delete the synced tables (registered in catalog_workshop.lakebase_<you>)
    for t in ["machines", "sensor_readings", "production_orders", "maintenance_tickets"]:
        try: w.postgres.delete_synced_table(name=f"synced_tables/{CATALOG}.{LAKEBASE_SCHEMA}.{t}")
        except Exception as e: print(t, e)

    # 3) drop the schema (synced replicas + lb_*_history)
    try: spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{LAKEBASE_SCHEMA} CASCADE")
    except Exception as e: print("schema:", e)

    # 4) delete the Lakebase project (drops the Postgres operational tables with it)
    if PROJECT:
        try: w.postgres.delete_project(name=f"projects/{PROJECT}"); print(f"deleted project {PROJECT}")
        except Exception as e: print("project:", e)
    print("done")
else:
    print("Teardown skipped — set RUN_TEARDOWN = True (and stop CDF first) to remove resources.")

## 🎉 Lab 4 Complete!

### What You Accomplished
✅ Read the `lb_*_history` **change-history** tables CDF produced
✅ Confirmed operator activity from the app lands in the lakehouse **automatically**
✅ Computed a real KPI (**time-to-fix / MTTR**) by joining operational writes to the original alerts

### The Round-Trip, Closed
**UC → Lakebase → app → Change Data Feed → back to UC** — one governed platform served both the analyst and the person on the floor, and their actions flowed straight back into analytics. No second database to secure, no bespoke ETL.